# Attention  Model Optimization Pipeline

In [5]:
import all_imports as ai

In [2]:
import importlib
import attention_model
importlib.reload(attention_model)



<module 'attention_model' from 'c:\\mythesis_work\\attention_model.py'>

In [5]:
from attetion_prepare import extract_X_y_M_delta, SequencePreprocessor
from attention_model import ModelBuilder, MSEPerTimestep, ModelEvaluator
from attention_model_plot import PredictionVisualizer, AttentionAnalyzer, TrainingPlotter

In [6]:
data = ai.pd.read_csv("clean_data.csv", index_col=0) 
data.index = ai.pd.to_datetime(data.index)

In [14]:
config = {
    "input_len": 24,          
    "output_len": 24,         
    "input_features": 18,    
    "target_features": 1,   
    "hidden_size": 128,
    "num_layers": 2,
    "dropout_rate": 0.3,
    "target_index": 0,  
    "deco_input_features": 1,   
    "epochs": 1,           
}

In [ ]:


def robust_objective(trial, train_seq, val_seq):
    """Robust objective function with proper value handling"""
    
    # 1. HYPERPARAMETER DEFINITION (Clear and organized)
    config = {
        # Architecture
        'units': trial.suggest_categorical("units", [64, 128, 256]),
        
        # Optimization
        'learning_rate': trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        'batch_size': trial.suggest_categorical("batch_size", [64, 128, 256]),
        
        # Regularization
        'input_dropout': trial.suggest_float("input_dropout", 0.0, 0.3),
        'recurrent_dropout': trial.suggest_float("recurrent_dropout", 0.0, 0.3),
        'output_dropout': trial.suggest_float("output_dropout", 0.0, 0.5),
        'clip_value': trial.suggest_float("clip_value", 0.5, 2.0),
        
        # Training
        'epochs': trial.suggest_int("epochs", 3, 25),
    }
    
    print(f"\n🎯 TRIAL {trial.number} STARTING")
    print("🔧 Hyperparameters:")
    for key, value in config.items():
        print(f"   {key}: {value}")
    
    # 2. MODEL BUILDING (With proper cleanup)
    ai.tf.keras.backend.clear_session()
    
    try:
        model_builder = ModelBuilder(
            train_seq, 
            units=config['units'], 
            config=config  # Pass all config properly
        )
        model = model_builder.build_model()
        
        # Verify model built correctly
        if model is None:
            raise ValueError("Model building failed - returned None")
            
    except Exception as e:
        print(f"❌ Model building failed: {str(e)}")
        return float('inf')
    
    # 3. MODEL COMPILATION (Clear and explicit)
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=config['learning_rate'],
        clipvalue=config['clip_value']
    )
    
    model.compile(
        optimizer=optimizer,
        loss={"forecast": MSEPerTimestep()},
        loss_weights={"forecast": 1.0}
    )
    
    # 4. DATA PREPARATION (Explicit and verified)
    train_inputs = {
        "X_enc": train_seq["X_enc"],
        "delta_enc": train_seq["delta_enc"], 
        "decoder_input": train_seq["decoder_input"],
        "mask_enc": train_seq["mask_enc"]
    }
    
    val_inputs = {
        "X_enc": val_seq["X_enc"],
        "delta_enc": val_seq["delta_enc"],
        "decoder_input": val_seq["decoder_input"], 
        "mask_enc": val_seq["mask_enc"]
    }
    
    train_targets = {"forecast": train_seq["decoder_target"]}
    val_targets = {"forecast": val_seq["decoder_target"]}
    
    train_sample_weight = {"forecast": train_seq["mask_dec"]}
    val_sample_weight = {"forecast": val_seq["mask_dec"]}
    
    # Verify data shapes
    print("📊 Data Verification:")
    print(f"   Train X_enc: {train_inputs['X_enc'].shape}")
    print(f"   Val X_enc: {val_inputs['X_enc'].shape}")
    print(f"   Train targets: {train_targets['forecast'].shape}")
    print(f"   Val targets: {val_targets['forecast'].shape}")
    
    # 5. CALLBACKS (Minimal and safe)
    callbacks = [
        ai.tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=5,
            restore_best_weights=True, 
            verbose=0
        ),
        ai.tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=0
        )
    ]
    
    # 6. TRAINING (With proper error handling)
    try:
        train_samples = train_seq["X_enc"].shape[0]
        steps_per_epoch = max(1, train_samples // config['batch_size'])
        
        print(f"🚀 Training: {train_samples} samples, {steps_per_epoch} steps/epoch")
        
        history = model.fit(
            train_inputs, 
            train_targets,
            batch_size=config['batch_size'], 
            epochs=config['epochs'],
            validation_data=(val_inputs, val_targets, val_sample_weight),
            sample_weight=train_sample_weight, 
            callbacks=callbacks, 
            verbose=1,  # Show progress for debugging
            shuffle=True
        )
        
        # 7. RESULTS VALIDATION
        if not history.history['val_loss']:
            raise ValueError("No validation loss recorded")
            
        final_train_loss = history.history['loss'][-1]
        final_val_loss = history.history['val_loss'][-1]
        best_val_loss = min(history.history['val_loss'])
        best_epoch = history.history['val_loss'].index(best_val_loss) + 1
        
        # Check for training issues
        if (ai.np.isnan(final_train_loss) or ai.np.isnan(final_val_loss) or
            final_val_loss > 1000 or best_val_loss > 1000):
            print(f"❌ Trial {trial.number} - Unstable training")
            return float('inf')
        
        print(f"✅ TRIAL {trial.number} COMPLETED")
        print(f"   Best val_loss: {best_val_loss:.4f} (epoch {best_epoch})")
        print(f"   Final val_loss: {final_val_loss:.4f}")
        print(f"   Epochs trained: {len(history.history['loss'])}")
        
        # Store comprehensive trial info
        trial.set_user_attr("config", config)
        trial.set_user_attr("best_val_loss", best_val_loss)
        trial.set_user_attr("final_val_loss", final_val_loss)
        trial.set_user_attr("final_train_loss", final_train_loss)
        trial.set_user_attr("epochs_trained", len(history.history['loss']))
        trial.set_user_attr("best_epoch", best_epoch)
        
        return best_val_loss
        
    except Exception as e:
        print(f"❌ Trial {trial.number} training failed: {str(e)}")
        return float('inf')
        
    finally:
        # 8. CLEANUP (Always execute)
        del model
        ai.tf.keras.backend.clear_session()

def robust_optimize(train_seq, val_seq, n_trials=5):
    """Robust optimization pipeline"""
    
    print(f"🚀 ROBUST OPTIMIZATION STARTING")
    print(f"📊 Dataset info:")
    print(f"   Train samples: {train_seq['X_enc'].shape[0]}")
    print(f"   Val samples: {val_seq['X_enc'].shape[0]}")
    print(f"   Input features: {train_seq['X_enc'].shape[-1]}")
    print(f"   Target length: {train_seq['decoder_target'].shape[1]}")
    
    study = ai.optuna.create_study(
        direction="minimize",
        study_name="robust_seq2seq_optimization"
    )
    
    def objective(trial):
        return robust_objective(trial, train_seq, val_seq)
    
    with ai.tqdm(total=n_trials, desc="🎯 Optimization Trials") as pbar:
        def callback(study, trial):
            pbar.update(1)
            if study.best_trial:
                current_best = study.best_value
                pbar.set_postfix({"Best Loss": f"{current_best:.4f}"})
                
                print(f"\n📊 TRIAL {trial.number} SUMMARY:")
                print(f"   Result: {trial.value:.4f}")
                if trial.state == optuna.trial.TrialState.COMPLETE:
                    if trial.number == study.best_trial.number:
                        print(f"   🏆 NEW BEST CONFIGURATION!")
                    else:
                        print(f"   Current best: {current_best:.4f}")
                print("-" * 50)
        
        study.optimize(objective, n_trials=n_trials, callbacks=[callback])
    
    return study

def robust_final_train(train_seq, val_seq, best_config):
    """Robust final training with best configuration"""
    
    print(f"\n🎯 FINAL MODEL TRAINING")
    print("🏆 Best Configuration:")
    for key, value in best_config.items():
        print(f"   {key}: {value}")
    
    ai.tf.keras.backend.clear_session()
    
    # Build final model
    model_builder = ModelBuilder(
        train_seq, 
        units=best_config['units'], 
        config=best_config
    )
    model = model_builder.build_model()
    
    optimizer = ai.tf.keras.optimizers.Adam(
        learning_rate=best_config['learning_rate'],
        clipvalue=best_config['clip_value']
    )
    
    model.compile(
        optimizer=optimizer,
        loss={"forecast": MSEPerTimestep()},
        loss_weights={"forecast": 1.0}
    )
    
    # Prepare data (same as before)
    train_inputs = {
        "X_enc": train_seq["X_enc"],
        "delta_enc": train_seq["delta_enc"], 
        "decoder_input": train_seq["decoder_input"],
        "mask_enc": train_seq["mask_enc"]
    }
    val_inputs = {
        "X_enc": val_seq["X_enc"],
        "delta_enc": val_seq["delta_enc"],
        "decoder_input": val_seq["decoder_input"], 
        "mask_enc": val_seq["mask_enc"]
    }
    train_sample_weight = {"forecast": train_seq["mask_dec"]}
    val_sample_weight = {"forecast": val_seq["mask_dec"]}
    train_targets = {"forecast": train_seq["decoder_target"]}
    val_targets = {"forecast": val_seq["decoder_target"]}
    
    # Enhanced callbacks for final training
    callbacks = [
        ai.tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=10,
            restore_best_weights=True, 
            verbose=1
        ),
        ai.tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        )
    ]
    
    # Training info
    train_samples = train_seq["X_enc"].shape[0]
    batch_size = best_config['batch_size']
    steps_per_epoch = max(1, train_samples // batch_size)
    best_epochs = best_config["epochs"]
    
    print(f"📊 Final Training Setup:")
    print(f"   - Batch size: {batch_size}")
    print(f"   - Steps per epoch: {steps_per_epoch}")
    print(f"   - Total epochs: {best_epochs}")
    print(f"   - Training samples: {train_samples}")
    
    print("🚀 Starting final training...")
    history = model.fit(
        train_inputs, train_targets,
        batch_size=batch_size, 
        epochs=best_epochs,
        validation_data=(val_inputs, val_targets, val_sample_weight),
        sample_weight=train_sample_weight, 
        callbacks=callbacks, 
        verbose=1
    )
    
    return model, history

def robust_optimization_pipeline(data, target_col, n_trials=5):
    """Complete robust optimization pipeline"""
    
    print("🚀 ROBUST OPTIMIZATION PIPELINE")
    print("=" * 50)
    
    # 1. Data Preparation
    print("📊 STEP 1: Preparing data...")
    X_seq, y_seq, M_seq, delta_seq, target_col = extract_X_y_M_delta(
        data, target_col, sparse_year=2020, start_hour=1, end_hour=7
    )
    
    preprocessor = SequencePreprocessor(
        input_seq_len=24, 
        output_seq_len=48, 
        target_col=target_col, 
        train_ratio=0.7, 
        val_ratio=0.15
    )
    train_seq, val_seq, test_seq, output_len, target_col = preprocessor.process(X_seq, y_seq, M_seq, delta_seq)
    
    print("✅ Data preparation completed")
    print(f"   Train sequences: {train_seq['X_enc'].shape[0]}")
    print(f"   Val sequences: {val_seq['X_enc'].shape[0]}")
    print(f"   Test sequences: {test_seq['X_enc'].shape[0]}")
    print(f"   Forecast horizon: {output_len}")
    
    # 2. Optimization
    print("\n🔄 STEP 2: Running hyperparameter optimization...")
    study = robust_optimize(train_seq, val_seq, n_trials=n_trials)
    
    # 3. Extract best configuration
    best_config = study.best_trial.user_attrs["config"]
    
    print(f"\n🎯 OPTIMIZATION RESULTS:")
    print("=" * 60)
    print(f"🏆 Best configuration from {n_trials} trials:")
    for key, value in best_config.items():
        print(f"   {key}: {value}")
    print(f"📈 Best validation loss: {study.best_value:.4f}")
    print("=" * 60)
    
    # 4. Final Training
    print("\n🎯 STEP 3: Training final model...")
    final_model, history = robust_final_train(train_seq, val_seq, best_config)
    
    # 5. Return comprehensive results
    results = {
        'best_config': best_config,
        'final_model': final_model,
        'training_history': history,
        'study': study,
        'best_validation_loss': study.best_value,
        'test_seq': test_seq,
        "forecast_horizon": output_len,
        "target_col": target_col,
        "scaler_y": test_seq["scaler_y"]
    }
    
    print("✅ ROBUST OPTIMIZATION PIPELINE COMPLETED!")
    
    return results

def get_robust_best_config(data, targets_col="CO", n_trials=1):
    """Get best configuration using robust pipeline"""
    results = robust_optimization_pipeline(data, target_col=targets_col, n_trials=n_trials)
    return results

In [ ]:
results = get_robust_best_config(data, "PMI-10", 5)

In [ ]:
results

In [ ]:

def evaluate_and_plot(results, plot_history=False, plot_erros=False, plot_attention=False):
    best_config, final_model, history, study, best_val_loss, test_seq, forecast_horizon, target_cols, scaler_y = (
    results["best_config"], results["final_model"], results["training_history"],
    results["study"], results["best_validation_loss"], results["test_seq"], results["forecast_horizon"], results["target_col"], results["scaler_y"])
    evaluator = ModelEvaluator(final_model, test_seq, scaler_y, forecast_horizon, "block_Sparsity_long_term", target_cols)
    results, outcome = evaluator.evaluate()

    targets_original = outcome["targets_original"]
    predictions_original = outcome["predictions_original"]
    mask = outcome["mask"]

    if plot_history == True:
        plotter = TrainingPlotter(target_cols, forecast_horizon)
        plotter.plot_loss_history(history, save_path=(fr"atten_plots\history_LT_{target_cols}_{forecast_horizon}"))
        plotter.plot_mae_history(history)

    if plot_erros ==True:
        visualizer = PredictionVisualizer(targets_original, predictions_original, mask, forecast_horizon, target_cols)
        visualizer.single_prediction_plot(sample_idx=0, save_path=(fr"atten_plots\predictions_block_LT_{target_cols}_{forecast_horizon}"))
        visualizer.scatter_plot(save_path=(fr"atten_plots\scatter_block_LT_{target_cols}_{forecast_horizon}"))
        visualizer.error_plot(save_path=(fr"atten_plots\errors_block_LT_{target_cols}_{forecast_horizon}"))
        visualizer.absolute_error_plot(save_path=(fr"atten_plots\absolute_error_block_LT_{target_cols}_{forecast_horizon}"))
        visualizer.residuals_plot(save_path=(fr"atten_plots\residuals_block_LT_plot_{target_cols}_{forecast_horizon}"))
        visualizer.error_percentage_plot(save_path=(fr"atten_plots\percentage_plot__block_LT_{target_cols}_{forecast_horizon}"))
    if plot_attention == True:
        attention_analysis = AttentionAnalyzer(final_model, test_seq, target_cols, forecast_horizon)
        heatmap = attention_analysis.attention_heatmap_plot(save_path=(fr"atten_plots\attention_heatmap__block_LT_{target_cols}_{forecast_horizon}"))
        attention_analysis.attention_timeline_plot(save_path=(fr"atten_plots\attention_timeline__block_LT_{target_cols}_{forecast_horizon}"))
        attention_analysis.attention_focus_plot(save_path=(fr"atten_plots\attention_focus__block_LT_{target_cols}_{forecast_horizon}"))
        attention_analysis.top_attended_steps_plot(save_path=(fr"atten_plots\attention_attended_steps__block_LT_{target_cols}_{forecast_horizon}"))
        attention_analysis.attention_context_plot(save_path=(fr"atten_plots\attentioncontext_plot__block_LT_{target_cols}_{forecast_horizon}"))
        attention_analysis.attention_statistics_plot(save_path=(fr"atten_plots\attention_statistics_plot__block_LT_{target_cols}_{forecast_horizon}"))
    
  
    
        # final_model.save(fr"save_models\attention_model{target_cols}_{forecast_horizon}.h5")

    
    return results, best_config
    
    
    

In [ ]:
metrice, best_config = evaluate_and_plot(results, plot_history=True, plot_erros=True, plot_attention=True)

In [196]:
metrice

{'mae': 3.4266,
 'rmse': 4.6672,
 'mape': 23.0221,
 'forecast_horizon': 48,
 'sparsity_type': 'block_Sparsity_long_term',
 'target_col': 'PMI-10'}

In [197]:
attention_metrice = {**metrice, **best_config} # Merge both dictionaries
# results_attention = ai.pd.DataFrame.from_dict(attention_metrice, orient="index").T 
# results_attention.to_csv("attention.csv", index=False)

In [198]:
attention_model = ai.pd.DataFrame.from_dict(attention_metrice, orient="index").T
attention_model.to_csv("attention.csv", mode="a", header=False, index=False)

In [3]:
atten_results = ai.pd.read_csv("attention.csv")

In [ ]:
atten_results